# Tutorial 3: Introduction to Genetic Algorithms
---
Faculty of Mathematics, Vienna, April 22, 2026.

Author: Sonia Foschiatti

---

### Learning Objectives
The goal of this laboratory is to apply the theory we have learned about Genetic Algorithms from Chapter 1 of the book of Pétrowski and Ben-Hamida "Evolutionary Algorithm" to a mathematical problem. 

In our code you will see how to implement the generic evolutionary algorithm, different selection and variation operators. At the end you will be able to appreciate the entire pipeline and to further experiment.

---
## 1. Setup and Imports

In [ ]:
# Uncomment this cell to install the libraries
#!pip install numpy matplotlib

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

print("Libraries imported successfully!")

## 2. Introduction

Genetic Algorithms are search algorithms that were studied by John Holland and his collaborators during the 1960s-1970s. They wanted to simulate the mechanisms of natural selection and natural genetics in computer science to understand and designe adaptive systems. This framework has also been proven successful for tackling complex optimization problems.

We will apply this theory to solve the following nonlinear problem: we want to maximize the continuous function

$$f(x) = 3x^4-5x^2+1 - 3\sin(5x)$$

for $x\in [-3,3]$.


The code we implement will make use of some terms deriving from the theory of evolution, that we summarize here:

| Natural Evolution | Genetic Algorithm |
|----------|----------|
| Genotype | coded string |
| Phenotype | uncoded string |
| Chromosome | string |
| Gene | string position |
| Allele | value at a certain position |
| Fitness | objective function value |

Genetic Algorithms are random processes. In order to maintain reproducibility, we fix a seed for the call of the random module.

We initialize also some parameters related with the algorithm. Among them, we fix the size of the population of chromosomes that we want to generate and the length of the chromosomes.

In [ ]:
SEED = 42
random.seed(SEED)

# global variables
popsize = 30
lengthChromosome = 64 # number of elements in a string
# interval extrema
xi, xf = -3, 3 

Genetic algorithms work with **strings** of bits, so we will have to define a function that allows us to convert `chromosome`-strings to real values using **binary encoding**. This will be used in the fitness function that we define later. 

Our function goes from Genotype to Phenotype and converts a string of bits into a binary integer and then scales it into the interval $[x_i,x_f]$ using the formula
$$ x = x_i + i\cdot \frac{x_f - x_i}{2^{\text{lengthChromosome}} - 1}.$$

In [ ]:
def decode(chromosome):
    # reorder the chromosome
    newChromosome = chromosome[::-1]
    numBin = 0
    powerof2 = 1
    # convert the chromosome using binary encoding
    for i,value in enumerate(newChromosome):
        if value != 0:
            numBin = numBin + powerof2
        powerof2 *= 2
    # scale
    phenotype = xi + numBin * (xf - xi) / (2**(len(chromosome)) - 1)
    return phenotype

In [ ]:
# Test our function with the two interval extrema
extrema1 = [0]*lengthChromosome
print(f"{decode(extrema1)}")

In [ ]:
extrema2 = [1]*lengthChromosome
print(f"{decode(extrema2)}")

We define a function that initialize a chromosome, namely a string of randomly chosen bits.

In [ ]:
def chromosome():
    return [random.randint(0,1) for i in range(lengthChromosome)]

In [ ]:
# test our function
chromosomeRand = chromosome()
print(chromosomeRand)
print(decode(chromosomeRand))

## 3. Fitness function


In [ ]:
def fitness(chromosome):
    phenotype = decode(chromosome)
    return 3*phenotype**4-5*phenotype**2 +1-3*np.sin(5*phenotype)


# Show the graph of the fitness function
x = np.linspace(-3,3,100)
y = 3*(x**4)-5*(x**2)+1-3*np.sin(5*x)
plt.plot(x,y, label="$f(x)=3x^4-5x^2+1-3sin(5x)$")
plt.title("Graph of the fitness function")
plt.legend()
plt.xlabel("x")
plt.axis([-3,3,-10,50])
plt.grid(axis="both")
plt.show()

**Question:** Where do we expect the maximum of this function to be?

In [ ]:
print(f"The fitness of {decode(chromosomeRand):.3g} is {fitness(chromosomeRand):.3g}")

We initialize the population and the vector with the fitness of each individual for testing later our functions.

In [ ]:
population = [chromosome() for i in range(popsize)]
fitnesses = [fitness(chrom) for chrom in population]

In [ ]:
# check 
print(f"Population length: {len(population)}, and type: {type(population)}.")

We introduce some statistics about the current fitness.

In [ ]:
print(f"The best fit is {max(fitnesses):.3g}.")
print(f"The average fitness is {np.mean(fitnesses):.3g}.")

## 4. Selection Operators

We will implement two selection operators: the Roulette Wheel Selection (RWS) and the Stochastic Universal Sampling (SUS) selection.

Recall that in the RWS we select the individual $i$ such that
$$\sum_{j=0}^{i-1}f(j)\leq r\sum_{j=1}^{\mu}f(j)\leq \sum_{j=1}^{i}f(j)$$
where $r$ is a random number drawn from $[0,1]$ and $\mu$ is the population size. Notice: the weigths in RWS are assumed to be positive. Therefore, we introduce a trick to make our algorithm valid.

In [ ]:
def roulette_wheel_selection(population, fitnesses, k):
    # rescaling fitness to have positive values
    minFitness = min(fitnesses)
    shiftedFitness = [fitness - minFitness + 1e-6 for fitness in fitnesses]
    total = float(sum(shiftedFitness))
    cumulative = np.cumsum(shiftedFitness)
    selected = []
    for j in range(k):
        x = total * np.random.random()
        for i, cum_f in enumerate(cumulative):
            if x < cum_f:
                selected.append(population[i])
                break
    return selected

In [ ]:
k = 6
selection1 = roulette_wheel_selection(population, fitnesses, k)
fitnesses_selection1 = [fitness(selection) for selection in selection1]
index_best_fit1 = int(np.argmax(fitnesses_selection1))
print(f"Selected {len(selection1)} individuals via RWS.")
print(f"The best fitness is {max(fitnesses_selection1):.3g}.")
print(f"The best fit is the value {decode(selection1[index_best_fit1]):.3g}")

In [ ]:
type(selection1)

In [ ]:
for individual in selection1:
    print(f"Individual: {decode(individual):.3g}, fitness: {fitness(individual):.3g}")

**Question:** Do you remember the problem with the RWS? Can we see it in the above solution?

Recall that in the SUS selection, a single random value is used to sample all the solutions. 

In [ ]:
def sus_selection(population, fitnesses, n):
    minFitness = min(fitnesses)
    shiftedFitness = [fitness - minFitness + 1e-6 for fitness in fitnesses]
    total = sum(shiftedFitness)
    # distance between the pointers
    delta = total / n
    omega = random.uniform(0,delta)
    pointers = [omega + i*delta for i in range(n)]
    # initiate the output 
    selected = []
    cumulative = np.cumsum(shiftedFitness)
    for pointer in pointers:
        for i, cum_f in enumerate(cumulative):
            if cum_f >= pointer:
                selected.append(population[i])
                break
    return selected

In [ ]:
# test our algorithm
n = 6
selection2 = sus_selection(population, fitnesses, n)
fitnesses_selection2 = [fitness(selection) for selection in selection2]
index_best_fit2 = int(np.argmax(fitnesses_selection2))
print(f"Selected {len(selection2)} individuals via SUS.")
print(f"The best fit is {max(fitnesses_selection2):.3g}.")
print(f"The best fit is the value {decode(selection2[index_best_fit2]):.3g}")

In [ ]:
for individual in selection2:
    print(f"Individual: {decode(individual):.3g}, fitness: {fitness(individual):.3g}")

## 5. Variation Operators

### Crossover
We implement here the so-called **single-point crossover**, where a cut-point is randomly chosen within the length of the individual. Other types are **two-point crossover**, **uniform crossover**, **blend crossover**, **ordered crossover**, and **partially mapped crossover**.

In [ ]:
def crossover(parent1, parent2, lengthString, pcross):
    # We initialize the offsprings
    offspring1, offspring2 = [], []
    # we toss a coin and check if its value is above the threshold pcross
    if np.random.random() < pcross:
        # selection of crossing site
        jcross = np.random.randint(1,lengthString-1)
    else:
        return parent1, parent2
    for j in range(jcross):
        offspring1.append(parent1[j])
        offspring2.append(parent2[j])
    for j in range(jcross,lengthString):
        offspring1.append(parent2[j])
        offspring2.append(parent1[j])
    
    return offspring1, offspring2

In [ ]:
# verify that Crossover works
parent1 = chromosome()
parent2 = chromosome()
offspring1, offspring2 = crossover(parent1, parent2, lengthString= lengthChromosome, pcross=0.8)
print(f"Parent 1: {decode(parent1):.3g} and Parent 2: {decode(parent2):.3g}")
print(f"Offspring 1: {decode(offspring1):.3g} and Offspring 2: {decode(offspring2):.3g}")

### Mutation 
We perform a **bit-flip** mutation, where a single gene is mutated when a certain prescribed threshold is overcome.

In [ ]:
def mutation(chromosome, pmutation):
    mutant = []
    for allele in chromosome:
        if np.random.random() < pmutation:
            mutant.append(1 - allele)
        else:
            mutant.append(allele)
    return mutant

In [ ]:
# test
chromosomeRandMut = mutation(chromosomeRand, pmutation=0.6)
print(f"Chromosome before mutation: {decode(chromosomeRand):.3g}")
print(f"Chromosome after mutation: {decode(chromosomeRandMut):.3g}")

## 6. Creation of a new population


In this step we are going to implement a function that places the parents with an offspring on which we perform selection, crossover and mutation operations. 

**Remark:** We assume that the population has an even number of elements.

In [ ]:
def create_offspring(old_population, selection_method, pcross, pmutation, popsize):
    """
    selection_method takes values "rws" and "sus"
    """
    new_population = []
    fitnesses = [fitness(chromosomeOld) for chromosomeOld in old_population]
    for i in range(0, popsize, 2):
        if selection_method == "rws":
            mate1, mate2 = roulette_wheel_selection(old_population, fitnesses, k=2)
        elif selection_method == "sus":
            mate1, mate2 = sus_selection(old_population, fitnesses, n=2)
        else: 
            raise ValueError(f"Unknown selection: {selection_method}")
        offspring1, offspring2 = crossover(mate1, mate2, len(mate1), pcross)
        offspring1 = mutation(offspring1,pmutation)
        offspring2 = mutation(offspring2,pmutation)
        new_population.extend([offspring1, offspring2])
    # in the case the popsize is odd
    if len(new_population) > popsize:
        new_population = new_population[:popsize]
    return new_population

## 7. Genetic Algorithm
Now we are ready to implement the whole pipeline of the genetic algorithm as an iterative algorithm for a fixed number of generations that performs the following operations:
- parental selection
- crossover + mutation
- creation of a new offspring and evaluation of its fitness

In [ ]:
# global variables
population_size = 100
max_generations = 100
pcross = 0.8
pmutation = 1/lengthChromosome

In [ ]:
# initialize the population
population = [chromosome() for i in range(population_size)] 

In [ ]:
# initialize some statistics
bestFitnessGenRWS = []
meanFitnessGenRWS = []
bestFitnessGenSUS = []
meanFitnessGenSUS = []


# generation loop for RWS
population_rws = population
for gen in range(max_generations):
    # 1. fitness evaluation
    fitnesses = [fitness(chromosome) for chromosome in population_rws]
    # statistics
    bestFitnessGenRWS.append(max(fitnesses))
    meanFitnessGenRWS.append(np.mean(fitnesses))
    
    # 2. Create the new offspring
    offspring_rws = create_offspring(population_rws, "rws", pcross, pmutation,population_size)   
    population_rws = offspring_rws
    
# generation loop for SUS
population_sus = population
for gen in range(max_generations):
    # 1. fitness evaluation
    fitnesses = [fitness(chromosome) for chromosome in population_sus]
    # statistics
    bestFitnessGenSUS.append(max(fitnesses))
    meanFitnessGenSUS.append(np.mean(fitnesses))
    
    # 2. Create the new offspring
    offspring_sus = create_offspring(population_sus, "sus", pcross, pmutation,population_size)
    population_sus = offspring_sus

**Remark:** Here we have artificially fixed a number of maximum generations without making a convergence check in advance. In real life, one can try other approaches to stop the algorithm, for example when the improvement in the best fitness falls below a certain threshold.

In [ ]:
print(f"The best fitness among generations is {max(bestFitnessGenRWS):.3g}")
print(f"The mean fitness among generations is {np.mean(meanFitnessGenRWS):.3g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(bestFitnessGenRWS, color="steelblue", label="RWS", linewidth=2)
axes[0].plot(bestFitnessGenSUS, color="seagreen", label="SUS", linewidth=2)
axes[1].plot(meanFitnessGenRWS, color="steelblue", label="RWS", linewidth=2)
axes[1].plot(meanFitnessGenSUS, color="seagreen", label="SUS", linewidth=2)

axes[0].set_title("Best fitness per generation")
axes[1].set_title("Mean fitness per generation")
for ax in axes:
    ax.set_xlabel("Generation")
    ax.set_ylabel("Fitness F(x)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

**Remark:** We have implemented a **full generational replacement** GA, so that the population is completely regenerated at each iteration. This has the disadvantage of losing the best global chromosome found in the next generation. A possible fix is **elitism**, for example $(1+\lambda)$ elitism: we keep the best fit and then we fit the population with the new offspring.

## Exercises (in class)
Look at the plots of the best fitness and the mean fitness.
1. Looking at the first plot, where is the lack of elitism evident?
2. Look again at the first plot: which method has a more stable convergence to the best fitness? Look at the second plot: which curve is smoother? What does this tell you about the sampling variance?
3. Write a `for` loop implementing the genetic algorithm for the two selection methods using the following values for the crossover probability `pcross`: `[0.0, 0.5, 0.9]`. Plot the results. Which value of `pcross` leads to a better convergence?

Work in groups, estimated time: 10 minutes.

In [ ]:
# write your code here

## Exercise (at home): 

1. Implement the deterministic Tournament Selection Operator without replacement in each tournament `p` (an individual cannot be chosen multiple times). Hint: to select the three indices, use the function `random.sample`: `candidate_indices = random.sample(range(len(population)),k=p)`
2. Create a new function `create_offspring2` which implements $(1+\lambda)$-elitism.
Hint: after the creation of the fitnesses list, create a copy of the fitnesses and call it `fitnessDecrease`. Then you can loop with length `elitism` over the list of chromosome until reaching the one realizing the maximum of the fitness, and save the corresponding chromosome in the new population. Then modify the length of the loop dedicated to implement the offspring accordingly. 
3. Run the Genetic Algorithm with elitism and the three selection methods: RWS, SUS and deterministic Tournament Selection for tournaments `p=10`. Plot the best fitness value and the mean of the fitnesses. Which method is more stable?
4. What can you say about the phenomenon of selection pressure?

In [ ]:
# write your code here
def tournament_selection(population, fitnesses, population_size, p):
    pass

In [ ]:
def create_offspring2(old_population, selection_method, pcross, pmutation, popsize, elitism):
    pass

## Optional Exercise
Implement a class called `SimpleGA` which implements:
- a `__init__` method that initialize the population size, the chromosome length, the continuous function to be studied, the interval extrema, the probability of crossover and mutation, elitism, maximum number of generations.
- a method that initialize the population
- a method that implement the fitness function
- a selection method
- a crossover method
- a mutation method
- a create_offrspring method
- a `run` method that implements the genetic loop. It should return the best individual and its fitness. 

Finally, test the code on the function introduced during the lecture.

In [ ]:
# you can use this draft
class SimpleGA:
    def __init__(self, fitness_fn, population_size=100, max_gen=100, lengthChromosome=8,
                 xi=-3, xf=3, pcross=0.8, pmutation=0.1, elitism=1, selection="rws"):
        pass
    
    # Here write the methods
    
    def run(self):
        # here write the code
        # .....
        final_fitnesses = [self.fitness(ch) for ch in pop]
        idx_best = int(np.argmax(final_fitnesses))
        return pop[idx_best], final_fitnesses[idx_best]

In [ ]:
ga = SimpleGA()
best_individual, best_fitness = ga.run()
# print the best individual, its fitness and decode it